# Soup GRPO task: step 1b, second pilot with longer answers (Kaggle T4)

**Before running:** Settings, then **Accelerator: GPU T4 x2** and **Internet: On**. Then **Save Version ->
Save & Run All (Commit)** and close the tab. Takes about **1 h 45 min**. When the version shows Complete, open
its **Output** tab and download **`soup_results.zip`**.

Why: in the first pilot, 65% of the 4-number answers were cut off at 256 tokens, so the pilot measured
"finishes in 256 tokens" more than arithmetic. The pre-registered rule says: raise the cap. This run uses
**512 new tokens** for 4 and 5 numbers (rule and predictions in `predictions/PREDICTIONS.md`, "Pilot 2"),
with the checker fixed in E11. Steps 1-5 are the same safety checks as before (8 GB cap, CUDA equivalence).

In [ ]:
# 1. Unpack code + data (git 92ddcae, bundle sha256 0fe55567c34a7343)
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
from pathlib import Path
LOCAL_TEST = os.environ.get("SOUP_LOCAL_TEST") == "1"   # only for testing this notebook on a laptop
WORK = Path(os.environ.get("SOUP_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "0fe55567c34a73436a72970440b8f03ed2ea3353fd1f42a994a6336cb122dab5", "bundle corrupted"
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(WORK, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
REPO = WORK / "soup-grpo"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
man = json.loads((REPO / "data/MANIFEST.json").read_text())
for f, info in man["files"].items():
    assert hashlib.sha256((REPO / "data" / f).read_bytes()).hexdigest() == info["sha256"], f
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
(REPO / "results/pilot").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| data files verified:", len(man["files"]), "| bundle git rev 92ddcae")

In [ ]:
# helper: run a command, stream its output live, keep a copy in results/logs
def run(cmd, log=None, check=True, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    t0 = time.time()
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    lines = []
    for line in p.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        lines.append(line)
    p.wait()
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[exit {p.returncode}, {time.time() - t0:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p.returncode

PY = sys.executable
STOP = None   # set by the first failing step; later steps are skipped, results are still packed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU, as in the task
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less cache fragmentation (pilot 1: 2 GiB)
MODEL = "Qwen/Qwen3-0.6B" if LOCAL_TEST else "Qwen/Qwen3-4B"
DEVICE = "cpu" if LOCAL_TEST else "cuda"

In [ ]:
# 2. Environment + dependencies
if not LOCAL_TEST:
    run([PY, "-m", "pip", "install", "-q", "transformers>=4.51", "safetensors>=0.4", "huggingface_hub"], check=False)
    TS = time.strftime("%Y%m%d-%H%M%S")
    env_txt = subprocess.run("nvidia-smi; nvidia-smi --query-gpu=name,memory.total,driver_version,"
                             "pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max "
                             "--format=csv; free -g; nproc; df -h /kaggle/working",
                             shell=True, capture_output=True, text=True).stdout
    import torch, transformers
    env_txt += f"\ntorch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__}"
    env_txt += f"\ndevice {torch.cuda.get_device_name(0)} capability {torch.cuda.get_device_capability(0)}"
    env_txt += f" bf16_supported {torch.cuda.is_bf16_supported()}"
    (REPO / f"results/logs/env_{TS}.txt").write_text(env_txt)
    print(env_txt)

In [ ]:
# 3. Session monitor (nvidia-smi + host RAM every second) and the 8 GB ballast
if STOP is None:
    try:
        if not LOCAL_TEST:
            TS = time.strftime("%Y%m%d-%H%M%S")
            smi_log = open(REPO / f"results/logs/nvsmi_{TS}.csv", "w")
            monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu,"
                                        "pcie.link.gen.current,pcie.link.width.current,temperature.gpu,power.draw,clocks.sm",
                                        "--format=csv", "-l", "1"], stdout=smi_log, stderr=subprocess.STDOUT)
            host_mon = subprocess.Popen(["bash", "-c", f"while true; do echo \"$(date -Is),$(free -m | awk '/Mem:/ {{print $3\",\"$7}}')\""
                                         f" >> {REPO}/results/logs/hostmem_{TS}.csv; sleep 1; done"])
            ballast = subprocess.Popen([PY, "scripts/gpu/ballast.py", "--cap-mib", "8192"],
                                       stdout=open(REPO / "results/logs/ballast.out", "w"), stderr=subprocess.STDOUT)
            for _ in range(120):
                if (REPO / "results/logs/ballast.json").exists():
                    break
                time.sleep(1)
            BALLAST = json.loads((REPO / "results/logs/ballast.json").read_text())
            print("ballast:", BALLAST)
            assert BALLAST["free_left_mib"] <= 8192, "cap not in place"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 4. Model download (~8 GB) and correctness of the engine on this GPU
if STOP is None:
    try:
        t0 = time.time()
        from soupgrpo.runtime import resolve_model_dir
        print("downloading", MODEL, "(~8 GB, a few minutes; no output until done) ...", flush=True)
        print("model at", resolve_model_dir(MODEL), f"({time.time() - t0:.0f}s)")
        run([PY, "-m", "pytest", "-q", "-x"], log="pytest.txt", check=False)
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/cuda_equivalence.py"], log="cuda_equivalence.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 5. PCIe bandwidth (pinned vs pageable, one 4B layer = 202 MB) + CUDA context cost
if STOP is None:
    try:
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/pcie_bench.py"], log="pcie.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 7. Pilot 2 (pre-registered in predictions/PREDICTIONS.md, "Pilot 2"): prompt v2, 512 new tokens, batch 48
if STOP is None:
    try:
        DTYPE = "fp32" if LOCAL_TEST else "fp16"
        LIMIT = ["--limit", "4"] if LOCAL_TEST else []
        MAXNEW = "48" if LOCAL_TEST else "512"
        def pilot(level, prompt="v2"):
            out = REPO / f"results/pilot/{level}_{prompt}_512"
            run([PY, "scripts/sample_eval.py", "--model", MODEL, "--device", DEVICE, "--dtype", DTYPE, "--prompt", prompt,
                 "--data", f"data/pilot_{level}.jsonl", "--samples", "4", "--temperature", "1.0",
                 "--max-new", MAXNEW, "--batch", "48", "--out", out, *LIMIT], log=f"pilot2_{level}.txt")
            return json.loads(out.with_suffix(".summary.json").read_text())
        acc = {}
        for level in ["n4", "n5"]:
            s = pilot(level)
            acc[level] = s["accuracy"]
            print(f"==> {level}/v2 @512: {s['accuracy']:.1%}, mixed {s['mixed_prompt_share']:.1%}, "
                  f"truncated {s['truncated_share']:.1%}, OOM splits {s.get('oom_splits')}")
        (REPO / "results/pilot/choice2.json").write_text(json.dumps({"prompt": "v2", "max_new": 512, "accuracy": acc}, indent=2))
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. Summary with the pre-registered selection rule + answers to read by hand
if STOP is None:
    try:
        run([PY, "scripts/pilot_summary.py", *sorted(map(str, (REPO / "results/pilot").glob("*.summary.json")))],
            log="pilot_summary.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Peak memory of THIS run (nvidia-smi used minus the ballast) and pack results (always runs)
if not LOCAL_TEST and "smi_log" in globals() and "BALLAST" in globals():
    import csv
    rows = list(csv.reader(open(smi_log.name)))[1:]
    used = [int(r[1].strip().split()[0]) for r in rows if len(r) > 1 and r[1].strip().split()[0].isdigit()]
    # subtract everything that was on the card before our processes started (ballast + driver baseline);
    # pilot 1 subtracted only the ballast and over-reported by the 107 MiB baseline (E12)
    peak = max(used) - BALLAST["smi_used_after_mib"]
    cap = {"smi_peak_total_mib": max(used), "smi_before_us_mib": BALLAST["smi_used_after_mib"],
           "ballast_used_mib": BALLAST["ballast_used_mib"], "our_peak_mib": peak,
           "cap_mib": BALLAST["cap_mib"], "under_cap": peak <= BALLAST["cap_mib"], "samples": len(used)}
    (REPO / "results/logs/cap_summary.json").write_text(json.dumps(cap, indent=2))
    print(cap)
import shutil
(REPO / "results/logs/stop_reason.txt").write_text(str(STOP))
print("STOP reason:", STOP)
zip_base = WORK / "soup_results"
shutil.make_archive(str(zip_base), "zip", REPO / "results")
print("DOWNLOAD:", str(zip_base) + ".zip")